# Describing the World with Numbers — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 4: Describing the World with Numbers](https://learnai.robennals.org/vectors). You'll build vectors in PyTorch, compute dot products, normalize to unit vectors, and build a simple neuron from scratch.

*New to PyTorch? Start with the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a quick intro to tensors.*

In [ ]:
import torch

## Our Toy Animals

We'll use the same nine animals as the chapter. Each is rated from 0 to 1 on six properties: **big**, **scary**, **hairy**, **cuddly**, **fast**, **fat**. Each animal is a vector — just a list of six numbers.

In [ ]:
animals = {
    'Bear':     torch.tensor([0.90, 0.85, 0.80, 0.50, 0.40, 0.75]),
    'Rabbit':   torch.tensor([0.10, 0.02, 0.60, 0.95, 0.70, 0.15]),
    'Shark':    torch.tensor([0.80, 0.95, 0.00, 0.00, 0.75, 0.20]),
    'Mouse':    torch.tensor([0.02, 0.05, 0.30, 0.40, 0.60, 0.10]),
    'Eagle':    torch.tensor([0.35, 0.60, 0.05, 0.02, 0.95, 0.05]),
    'Elephant': torch.tensor([0.98, 0.30, 0.05, 0.40, 0.15, 0.95]),
    'Snake':    torch.tensor([0.20, 0.85, 0.00, 0.02, 0.50, 0.05]),
    'Cat':      torch.tensor([0.15, 0.30, 0.75, 0.85, 0.70, 0.25]),
    'Dog':      torch.tensor([0.45, 0.20, 0.70, 0.90, 0.55, 0.45]),
}

print(animals['Bear'])

## The Dot Product — A Measure of Similarity

To measure how similar two animals are, multiply each matching pair of numbers and add up the results. This is the **dot product**:

`dot(a, b) = a[0]*b[0] + a[1]*b[1] + ... + a[n-1]*b[n-1]`

When both animals score high on the same property, that product is big. When one scores high and the other low, the product is small. Add them up and you get a single number — high if the two animals agree, low if they don't.

In [ ]:
bear = animals['Bear']
dog = animals['Dog']

# Multiply matching pairs and add them up
manual = sum(bear[i] * dog[i] for i in range(6))
print(f'Manual:    {manual:.3f}')

# torch.dot does the same thing in one call
print(f'torch.dot: {torch.dot(bear, dog):.3f}')

### Comparing the Whole Zoo

Let's rank every animal by how similar it is to a bear.

In [ ]:
scores = {name: torch.dot(bear, vec).item() for name, vec in animals.items()}

for name, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f'{name:10s} {score:.3f}')

## The Problem with Raw Dot Products

The dot product rewards *having big numbers* — not just *agreeing on the right properties*. An animal that scored high on every property would look similar to everything. Let's add a fake `MaxAnimal` with 1.0 on every property and see what happens.

In [ ]:
animals_plus_max = {**animals, 'MaxAnimal': torch.ones(6)}
scores = {name: torch.dot(bear, vec).item() for name, vec in animals_plus_max.items()}

for name, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f'{name:10s} {score:.3f}')

## Magnitude — the Size of a Vector

To fix the fairness problem we need to put every vector on equal footing. Step one: measure a vector's **size**, or **magnitude**. Square each component, add them up, take the square root:

`magnitude(v) = sqrt(v[0]**2 + v[1]**2 + ... + v[n-1]**2)`

This is the Pythagorean rule, extended to any number of dimensions.

In [ ]:
magnitude_of_bear = torch.sqrt(torch.sum(bear ** 2))
print(f'Bear magnitude: {magnitude_of_bear:.3f}')

In [ ]:
def magnitude(v):
    return torch.sqrt(torch.sum(v ** 2))

def normalize(v):
    return v / magnitude(v)

## Unit Vectors

A **unit vector** has magnitude 1. To turn any vector into its unit vector, divide every component by the vector's magnitude. The *proportions* stay the same (so a bear is still scarier than it is fast), but the size is standardized.

In [ ]:
bear_unit = normalize(bear)
print(f'bear_unit:            {bear_unit}')
print(f'magnitude(bear_unit): {magnitude(bear_unit):.3f}')

# The decomposition is reversible: unit vector * magnitude == original
reconstructed = bear_unit * magnitude(bear)
print(f'reconstructed == bear? {torch.allclose(reconstructed, bear)}')

## Cosine Similarity

The dot product of two *unit vectors* is called **cosine similarity**. It ranges from −1 (opposite) through 0 (unrelated) to 1 (identical). Because both inputs have magnitude 1, nobody gets unfair credit for being big.

Watch what happens to `MaxAnimal` now.

In [ ]:
animals_unit = {name: normalize(vec) for name, vec in animals_plus_max.items()}
bear_unit = animals_unit['Bear']

scores = {name: torch.dot(bear_unit, vec).item() for name, vec in animals_unit.items()}

for name, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f'{name:10s} {score:.3f}')

## Blending Vectors

Add two vectors (component by component) and you get a new vector that mixes both. Blend a bear and a rabbit — which real animal is the blend closest to?

In [ ]:
rabbit = animals['Rabbit']
blend = normalize(bear + rabbit)

animals_unit = {name: normalize(vec) for name, vec in animals.items()}
scores = {name: torch.dot(blend, vec).item() for name, vec in animals_unit.items()}

print('Closest animals to a bear + rabbit blend:')
for name, score in sorted(scores.items(), key=lambda x: -x[1])[:5]:
    print(f'{name:10s} {score:.3f}')

## Modifying a Vector by Adding a Direction

You can also push a vector in a specific direction. Start with a bear and add *smaller, cuddlier, less scary*. What animal is that?

The six properties are ordered: big, scary, hairy, cuddly, fast, fat. So our direction vector is `[-0.4, -0.5, 0.0, +0.4, 0.0, -0.3]`.

In [ ]:
direction = torch.tensor([-0.4, -0.5, 0.0, 0.4, 0.0, -0.3])
modified = normalize(bear + direction)

scores = {name: torch.dot(modified, vec).item() for name, vec in animals_unit.items()}

print('Closest animals to bear + (smaller, cuddlier, less scary):')
for name, score in sorted(scores.items(), key=lambda x: -x[1])[:5]:
    print(f'{name:10s} {score:.3f}')

## A Neuron Is a Dot Product

Here's the punchline. A neuron computes: `output = activation(dot(weight, input) + bias)`.

- The **weight vector's direction** is what the neuron looks for.
- The **weight vector's magnitude** is how sensitive the neuron is.
- The **bias** shifts where it fires.
- The **activation function** squashes the result into a bounded range.

Let's build a **bear detector**. Set the weight's direction to the bear unit vector, scale it up (so the sigmoid makes sharper distinctions), and tune the bias so only strongly bear-like animals light up.

In [ ]:
weight = normalize(bear) * 10.0   # direction: bear; magnitude: 10 (sensitivity)
bias = -8.0                         # fires only for strong bear-likeness

print(f"{'Animal':10s} {'score':>7s} {'output':>7s}")
print('-' * 28)
for name, vec in animals.items():
    vec_unit = normalize(vec)
    score = torch.dot(weight, vec_unit) + bias
    output = torch.sigmoid(score)
    print(f'{name:10s} {score:+7.3f} {output:7.3f}')

## What's Next

We've seen vectors as lists of numbers, the dot product as similarity, unit vectors as fair-comparison scaffolding, vector addition as blending and modification, and a neuron as a dot product.

But we hand-picked the six properties. Next, [Chapter 5: From Words to Meanings](https://learnai.robennals.org/embeddings) shows how AI *learns* its own vectors — for things like words, where no human-chosen dimensions could possibly work.